# Prototype Demo Notebook

This retained-scope prototype demo now walks through two sealed local workflows: the Phase 06 probabilistic inspection surfaces and the retained Phase 07 local publication catalog. It runs sample manifests, resolves stored forecast artifacts, and then publishes a replay-verified retained run into a read-only catalog view.

The setup cell below pins the repository root, imports the public Python API, and points each workflow at the checked-in Phase 06 probabilistic manifests plus the retained publication demo manifest.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError("Run this notebook from the Euclid repository root.")

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from euclid import (
    inspect_demo_catalog_entry,
    inspect_demo_calibration,
    inspect_demo_probabilistic_prediction,
    load_demo_publication_catalog,
    publish_demo_run_to_catalog,
    run_demo,
    run_demo_probabilistic_evaluation,
)

MANIFEST_DIR = PROJECT_ROOT / "fixtures/runtime/phase06"
RETAINED_MANIFEST = PROJECT_ROOT / "fixtures/runtime/prototype-demo.yaml"
MANIFESTS = {
    "distribution": PROJECT_ROOT / "fixtures/runtime/phase06/probabilistic-distribution-demo.yaml",
    "interval": PROJECT_ROOT / "fixtures/runtime/phase06/probabilistic-interval-demo.yaml",
    "quantile": PROJECT_ROOT / "fixtures/runtime/phase06/probabilistic-quantile-demo.yaml",
    "event_probability": PROJECT_ROOT / "fixtures/runtime/phase06/probabilistic-event-probability-demo.yaml",
}
OUTPUT_ROOT = PROJECT_ROOT / "build/notebook-demo"
{
    "manifest_paths": {name: str(path) for name, path in MANIFESTS.items()},
    "output_root": str(OUTPUT_ROOT),
}


In [ ]:
probabilistic_cases = {}
manifest_paths = {
    "distribution": MANIFEST_DIR / "probabilistic-distribution-demo.yaml",
    "interval": MANIFEST_DIR / "probabilistic-interval-demo.yaml",
    "quantile": MANIFEST_DIR / "probabilistic-quantile-demo.yaml",
    "event_probability": MANIFEST_DIR / "probabilistic-event-probability-demo.yaml",
}
for forecast_object_type, manifest_path in manifest_paths.items():
    case_output_root = OUTPUT_ROOT / forecast_object_type
    run_result = run_demo_probabilistic_evaluation(
        manifest_path=manifest_path,
        output_root=case_output_root,
    )
    prediction = inspect_demo_probabilistic_prediction(output_root=case_output_root)
    calibration = inspect_demo_calibration(output_root=case_output_root)
    probabilistic_cases[forecast_object_type] = {
        "run": run_result,
        "prediction": prediction,
        "calibration": calibration,
        "artifact_summary": {
            "forecast_object_type": prediction.forecast_object_type,
            "score_law_id": prediction.score_law_id,
            "row_count": prediction.row_count,
            "aggregated_primary_score": prediction.aggregated_primary_score,
            "diagnostic_count": len(calibration.diagnostics),
            "calibration_status": calibration.status,
        },
    }

distribution_summary = probabilistic_cases["distribution"]["artifact_summary"]
interval_summary = probabilistic_cases["interval"]["artifact_summary"]
quantile_summary = probabilistic_cases["quantile"]["artifact_summary"]
event_probability_summary = probabilistic_cases["event_probability"]["artifact_summary"]
probabilistic_cases


The retained publication flow stays read-only against the sealed artifacts. The cell below runs the retained demo once, projects that replay-verified result into `catalog/index.json`, and resolves the published entry back through the public browse helpers.

In [ ]:
retained_output_root = OUTPUT_ROOT / "retained-catalog"
retained_run = run_demo(
    manifest_path=RETAINED_MANIFEST,
    output_root=retained_output_root,
)
published_entry = publish_demo_run_to_catalog(output_root=retained_output_root)
published_catalog = load_demo_publication_catalog(output_root=retained_output_root)
published_inspection = inspect_demo_catalog_entry(
    output_root=retained_output_root,
    publication_id=published_entry.publication_id,
)
{
    "run_result_ref": str(retained_run.summary.run_result_ref),
    "publication_id": published_entry.publication_id,
    "catalog_entries": published_catalog.entry_count,
    "publication_mode": published_entry.publication_mode,
    "bundle_ref": str(published_inspection.replay_bundle.bundle_ref),
}
